# 🛰️ Real Satellite Images — Sri Lanka Floods, 29–30 November 2025

Downloads **Sentinel‑2 true‑colour** satellite tiles over the flood‑affected
**Greater Colombo / Western Province** region (Cyclone *Ditwah*, 29–30 Nov 2025),
ready for training the GeoRescue flood‑vision model.

### ⚠️ Read this before running — 3 things

1. **Resolution.** Free Sentinel‑2 is **10 m / pixel**. You will clearly see the
   **road network, rivers, city blocks and large buildings** — but **not** single
   houses or cars. The crisp zoomed‑in picture you saw in EO Browser is a separate
   high‑res *basemap*, not the 29‑Nov data. Sub‑metre imagery needs a paid provider
   (Maxar / Planet); your `client_id`/`client_secret` are for Sentinel Hub.

2. **Clouds.** During a cyclone the sky is cloudy, so the exact 29–30 Nov optical
   pass may be partly hidden. This notebook searches a **date window** (25 Nov →
   5 Dec), keeps the **least‑cloudy** pixels, and prints a cloud report. For a
   guaranteed all‑weather flood map, run the **Sentinel‑1 SAR** cell at the bottom
   (radar sees *through* cloud).

3. **Two account types** — set `ACCOUNT` in the credentials cell:
   * `"cdse"` → **Copernicus Data Space Ecosystem** (free) — `sh.dataspace.copernicus.eu`
   * `"commercial"` → **Sentinel Hub** trial/paid — `services.sentinel-hub.com`

**How to use:** Runtime ▸ Run all, the tiles are saved into the Colab workspace (`/content/...`) and previewed
inline for inspection — nothing is auto-downloaded. Use the 📁 Files panel to
download them yourself once you like what you see.


In [ ]:
# 1️⃣  Install dependencies  (~30 s)
!pip -q install "sentinelhub>=3.10.0" pillow tqdm pandas matplotlib
print("✅ dependencies installed")

In [ ]:
# 2️⃣  Credentials (from Colab Secrets) + auto-detect server  ───────────────────
# Stored in the Colab "🔑 Secrets" panel as SENTINEL_CLIENT_ID / SENTINEL_CLIENT_SECRET.
# IMPORTANT: in that panel, toggle "Notebook access" ON for BOTH secrets.
import requests
from sentinelhub import SHConfig

try:
    from google.colab import userdata
    CLIENT_ID     = userdata.get("SENTINEL_CLIENT_ID")
    CLIENT_SECRET = userdata.get("SENTINEL_CLIENT_SECRET")
    print("🔑 loaded credentials from Colab Secrets")
except Exception as e:
    from getpass import getpass
    print("Could not read Colab Secrets (", e, ") — type them manually instead:")
    CLIENT_ID     = input("Client ID: ")
    CLIENT_SECRET = getpass("Client Secret (hidden): ")

# strip stray spaces/quotes that break OAuth
CLIENT_ID     = (CLIENT_ID or "").strip().strip('"').strip("'")
CLIENT_SECRET = (CLIENT_SECRET or "").strip().strip('"').strip("'")
print(f"   id length={len(CLIENT_ID)}, secret length={len(CLIENT_SECRET)}")

# Sentinel Hub has two deployments — try both, keep whichever authenticates.
ENDPOINTS = {
    "cdse (free)":   ("https://sh.dataspace.copernicus.eu",
                      "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"),
    "commercial":    ("https://services.sentinel-hub.com",
                      "https://services.sentinel-hub.com/oauth/token"),
}

config, ACCOUNT = None, None
for name, (base, token_url) in ENDPOINTS.items():
    try:
        r = requests.post(token_url, data={"grant_type": "client_credentials",
                                           "client_id": CLIENT_ID, "client_secret": CLIENT_SECRET},
                          timeout=30)
        print(f"  {name:13s} → HTTP {r.status_code}")
    except Exception as e:
        print(f"  {name:13s} → network error: {e}"); continue
    if r.status_code == 200:
        config = SHConfig()
        config.sh_client_id, config.sh_client_secret = CLIENT_ID, CLIENT_SECRET
        config.sh_base_url, config.sh_token_url = base, token_url
        config.max_download_attempts = 5
        config.download_sleep_time = 5
        config.download_timeout_seconds = 120
        ACCOUNT = name
        print(f"\n✅ authenticated on '{name}'  →  {base}")
        break

if config is None:
    raise SystemExit(
        "\n❌ Both servers rejected these credentials (401 invalid_client).\n"
        "   The Client ID/Secret are mistyped, expired, or from a different account.\n"
        "   Fix: open your Sentinel Hub dashboard → User settings → OAuth clients,\n"
        "   create a new client, and update the SENTINEL_CLIENT_ID / SENTINEL_CLIENT_SECRET\n"
        "   secrets in Colab's 🔑 panel (no quotes, no trailing spaces).")

In [ ]:
# 4️⃣  Area, dates & ZOOM / QUALITY  ────────────────────────────────────────────
# Tighter AOI on the worst-hit flood core: Colombo metro + Kelani river basin
# (Kolonnawa, Kelaniya, Wattala, Kaduwela). Smaller area = more zoomed-in tiles.
# Format: [lon_min, lat_min, lon_max, lat_max]
AOI_BBOX = [79.86, 6.86, 80.10, 7.05]

# Cyclone Ditwah flood window (searched as a range — clouds need leeway)
DATE_FROM = "2025-11-25"
DATE_TO   = "2025-12-05"

# ── Zoom & quality knobs ──────────────────────────────────────────────────────
TILE_KM    = 1.5     # ground size per tile (km).  SMALLER = more zoomed in.
RESOLUTION = 4       # output metres/pixel. Sentinel-2 is natively 10 m; 4 upsamples
                     # to a bigger, crisper-looking tile (~375 px). Don't go below 3.
MAX_CLOUD  = 40      # % — stricter = cleaner pictures (fewer usable dates)

print(f"AOI {AOI_BBOX} | ~{TILE_KM} km tiles @ {RESOLUTION} m/px | {DATE_FROM}->{DATE_TO}")

In [ ]:
# 5️⃣  Connect to the collection & list available passes (cloud report)  ───────
from sentinelhub import SentinelHubCatalog, BBox, CRS, DataCollection

def sh_collection(base, name, service_url):
    """Bind a data collection to the chosen server; safe on any account & re-runs."""
    try:
        return DataCollection.define_from(base, name, service_url=service_url)
    except ValueError:
        try:
            return DataCollection[name]   # we defined it on a previous run
        except KeyError:
            return base                   # URL already matches the built-in collection

S2 = sh_collection(DataCollection.SENTINEL2_L2A, "S2L2A_SH", config.sh_base_url)

catalog = SentinelHubCatalog(config=config)
aoi = BBox(AOI_BBOX, crs=CRS.WGS84)

scenes = list(catalog.search(
    S2, bbox=aoi, time=(DATE_FROM, DATE_TO),
    fields={"include": ["id", "properties.datetime", "properties.eo:cloud_cover"], "exclude": []},
))
print(f"{len(scenes)} Sentinel-2 passes between {DATE_FROM} and {DATE_TO}:\n")
for s in sorted(scenes, key=lambda x: x["properties"]["datetime"]):
    cc = s["properties"].get("eo:cloud_cover", "n/a")
    print(f"  {s['properties']['datetime'][:16]}   cloud {cc:>5}%")

In [ ]:
# 6️⃣  Build the tile grid over the AOI  ───────────────────────────────────────
import numpy as np
lon_min, lat_min, lon_max, lat_max = AOI_BBOX
KM_PER_DEG_LAT = 110.57
KM_PER_DEG_LON = 111.32 * np.cos(np.radians((lat_min + lat_max) / 2))

cols = max(1, round((lon_max - lon_min) * KM_PER_DEG_LON / TILE_KM))
rows = max(1, round((lat_max - lat_min) * KM_PER_DEG_LAT / TILE_KM))
dlon = (lon_max - lon_min) / cols
dlat = (lat_max - lat_min) / rows

tiles = [[round(lon_min + c*dlon, 6), round(lat_min + r*dlat, 6),
          round(lon_min + (c+1)*dlon, 6), round(lat_min + (r+1)*dlat, 6)]
         for r in range(rows) for c in range(cols)]
print(f"{cols} × {rows} = {len(tiles)} tiles (some sea tiles will be skipped on download)")

In [ ]:
# 7️⃣  Evalscripts — how each pixel is rendered  ───────────────────────────────
# True colour (B04,B03,B02), gamma-corrected for brighter/crisper tiles + valid mask.
EVALSCRIPT_TRUECOLOR = """
//VERSION=3
function setup() {
  return { input: [{ bands: ["B04","B03","B02","dataMask"] }], output: { bands: 4 } };
}
function adj(v) { return Math.max(0, Math.min(1, Math.pow(v * 3.0, 0.85))); }
function evaluatePixel(s) {
  return [adj(s.B04), adj(s.B03), adj(s.B02), s.dataMask];
}
"""

# Flood highlight — open water turns blue (extra training signal for flood extent).
EVALSCRIPT_FLOOD = """
//VERSION=3
function setup() {
  return { input: [{ bands: ["B03","B04","B08","dataMask"] }], output: { bands: 4 } };
}
function evaluatePixel(s) {
  let ndwi = (s.B03 - s.B08) / (s.B03 + s.B08);      // water index
  if (ndwi > 0.0) return [0.0, 0.4, 1.0, s.dataMask]; // water = blue
  return [2.5*s.B04, 2.5*s.B08, 2.5*s.B03, s.dataMask]; // false-colour land
}
"""
print("evalscripts ready")

In [ ]:
# 8️⃣  Download function  ──────────────────────────────────────────────────────
import csv
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from sentinelhub import SentinelHubRequest, MimeType, bbox_to_dimensions

def download_tiles(evalscript, collection, out_dir, *, date_from=DATE_FROM, date_to=DATE_TO,
                   resolution=RESOLUTION, max_cloud=MAX_CLOUD, min_valid_frac=0.30,
                   mosaicking="leastCC", cloud_filter=True):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    meta, saved = [], 0
    for i, tb in enumerate(tqdm(tiles, desc=out_dir.name)):
        sh_bbox = BBox(tb, crs=CRS.WGS84)
        size = bbox_to_dimensions(sh_bbox, resolution=resolution)
        size = (min(size[0], 2500), min(size[1], 2500))
        req = SentinelHubRequest(
            evalscript=evalscript,
            input_data=[SentinelHubRequest.input_data(
                data_collection=collection,
                time_interval=(date_from, date_to),
                mosaicking_order=mosaicking,
                other_args={"dataFilter": {"maxCloudCoverage": max_cloud}} if cloud_filter else {},
            )],
            responses=[SentinelHubRequest.output_response("default", MimeType.PNG)],
            bbox=sh_bbox, size=size, config=config,
        )
        try:
            arr = req.get_data()[0]
        except Exception as e:
            meta.append([i, *tb, "ERROR", str(e)[:80]]); continue

        rgb = arr[..., :3]
        valid = (arr[..., 3] > 0).mean() if arr.shape[-1] == 4 else (rgb.sum(-1) > 0).mean()
        if valid < min_valid_frac:                      # mostly sea / off-swath
            meta.append([i, *tb, "SKIP_empty", f"valid={valid:.2f}"]); continue

        fname = f"{out_dir.name}_{i:04d}_{tb[0]:.3f}_{tb[1]:.3f}.png"
        Image.fromarray(rgb.astype("uint8")).save(out_dir / fname)
        meta.append([i, *tb, fname, f"valid={valid:.2f}"]); saved += 1

    with open(out_dir / "metadata.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["idx", "lon_min", "lat_min", "lon_max", "lat_max", "file", "note"])
        w.writerows(meta)
    print(f"✅ saved {saved} images → {out_dir}  (metadata.csv written)")
    return saved

In [ ]:
# 9️⃣  Run the download (true colour) — FRESH folder each run
import shutil, glob
from datetime import datetime

# Delete previous run folders so images never pile up (set False to keep them).
CLEAN_PREVIOUS = True
if CLEAN_PREVIOUS:
    olds = ["/content/sri_lanka_floods_2025"] + glob.glob("/content/sri_lanka_floods_2025_*")
    for old in sorted(set(olds)):
        if old.endswith("_SAR"):            # keep the radar set, if any
            continue
        shutil.rmtree(old, ignore_errors=True)
        print("🗑️  removed old run:", old)

# New timestamped folder = ONLY this run's images live here.
RUN_ID  = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path(f"/content/sri_lanka_floods_2025_{RUN_ID}")
print("📂 saving this run to:", OUT_DIR)

download_tiles(EVALSCRIPT_TRUECOLOR, S2, OUT_DIR)

# Tip: blue-water flood version → download_tiles(EVALSCRIPT_FLOOD, S2, str(OUT_DIR) + "_flood")

In [ ]:
# 🔟  Inspect — preview what's now SAVED in the Colab workspace
#     Nothing is downloaded to your computer. The PNGs + metadata.csv live in
#     /content/sri_lanka_floods_2025. Open the 📁 Files panel (left sidebar) to
#     browse every image and download manually once you're happy.
import glob, matplotlib.pyplot as plt
from PIL import Image
files = sorted(glob.glob(str(OUT_DIR / "*.png")))
print(f"{len(files)} images saved in {OUT_DIR}")
print(f"→ Left sidebar 📁 Files ▸ {OUT_DIR.name}  (right-click a file/folder ▸ Download)")
n = min(30, len(files)); ncol = 5; nrow = (n + ncol - 1) // ncol
plt.figure(figsize=(15, 3 * nrow))
for k in range(n):
    plt.subplot(nrow, ncol, k + 1)
    plt.imshow(Image.open(files[k])); plt.axis("off")
    plt.title(Path(files[k]).name[-20:], fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# 1️⃣1️⃣  Download THIS run as a .zip  — run this cell whenever you want it
#     The zip is named after the run's timestamped folder, so every download is a
#     DIFFERENT file (e.g. sri_lanka_floods_2025_20260629_141530.zip) and they
#     never overwrite each other on your computer.
import shutil
from google.colab import files as colab_files

zip_path = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)   # unique name per run
print("📦 zipped:", zip_path)
colab_files.download(zip_path)                                 # triggers the browser download

---
## ⭐ Optional — Sentinel‑1 SAR (all‑weather flood map)

Radar **sees through clouds**, so this works even on the cloudiest cyclone day.
In SAR, calm flood water looks **dark/smooth**. Run the cell below only if you
want the true 29–30 Nov flood extent regardless of cloud.


In [ ]:
# Sentinel-1 SAR — water shows dark (low backscatter)
S1 = sh_collection(DataCollection.SENTINEL1_IW, "S1IW_SH", config.sh_base_url)
EVALSCRIPT_S1 = """
//VERSION=3
function setup() { return { input: ["VV","VH","dataMask"], output: { bands: 3 } }; }
function evaluatePixel(s) {
  let vv = Math.min(1, s.VV * 2.0);
  let vh = Math.min(1, s.VH * 5.0);
  return [vv, vh, vv];           // smooth water -> dark
}
"""
download_tiles(EVALSCRIPT_S1, S1, "/content/sri_lanka_floods_2025_SAR",
               cloud_filter=False, mosaicking="mostRecent")